# CIVID — Master Dashboard
### Civilian Impact Verified Incident Dataset (Phases 1–4)

**Maintainer / Author:** Muhammad Farhan

**What this notebook does**
- Runs the full CIVID pipeline end-to-end (validate → renumber → build exports → fetch leads).
- Reads the regenerated `exports/` (combined, ML/dashboard-ready CSVs) for all phases.
- Produces overview stats, timeline, verification/confidence/role breakdowns, source-reliability,
  famous-victims status, media/ethics status, and quality-flag summaries.

**Ethics note:** Aggregate figures and quality flags only. No victim or casualty imagery.
No identifiable images of minors. See `DATA_LICENSE.md` and `docs/usage_disclaimer.md`.


## 0. Full-system run & check
Run this cell to validate, renumber, rebuild exports, and fetch daily leads. It prints a go/no-go status for every step.

In [1]:
import subprocess, sys, os, json, pathlib
from IPython.display import Markdown, display

ROOT = pathlib.Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

def run(script, *extra):
    p = subprocess.run([sys.executable, str(ROOT / "scripts" / script), *extra],
                       capture_output=True, text=True)
    ok = p.returncode == 0
    tag = "OK " if ok else "ERR"
    print(f"[{tag}] python scripts/{script} {' '.join(extra)}  (exit={p.returncode})")
    out = (p.stdout or "") + (p.stderr or "")
    for line in out.splitlines():
        if "error" in line.lower() or "EXIT=" in line or "0 error" in line:
            print("      ", line)
    return ok

print("=== CIVID full-system run ===  (root:", ROOT, ")")
steps = [
    ("validate_dataset.py",),
    ("renumber_records.py",),
    ("build_exports.py",),
    ("fetch_leader_photos.py",),
    ("daily_update.py",),
]
results = {s[0]: run(*s) for s in steps}
all_ok = all(results.values())
summary_path = ROOT / "exports" / "summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("\nCombined snapshot:", json.dumps(summary.get("totals", {}), indent=2))
display(Markdown("### ✅ All systems go — proceed to the dashboard below."
                 if all_ok else
                 "### ⚠️ One or more steps failed — review the errors above before trusting outputs."))


=== CIVID full-system run ===  (root: g:\CIVID (Civilian Impact Verified Incident Dataset) )
[OK ] python scripts/validate_dataset.py   (exit=0)
         0 warning(s), 0 error(s)
[OK ] python scripts/renumber_records.py   (exit=0)
[OK ] python scripts/build_exports.py   (exit=0)
[OK ] python scripts/daily_update.py   (exit=0)

Combined snapshot: {
  "events": 63,
  "events_verified": 48,
  "events_unverified": 8,
  "persons": 14,
  "media": 0,
  "famous_victims": 0,
  "news_intelligence": 52,
  "images_indexed": 0,
  "needs_review": 8
}


### ✅ All systems go — proceed to the dashboard below.

## 1. Load regenerated exports

In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

EXPORTS = ROOT / "exports"
events  = pd.read_csv(EXPORTS / "civid_events_all.csv")
persons = pd.read_csv(EXPORTS / "civid_persons_all.csv")
famous  = pd.read_csv(EXPORTS / "civid_famous_victims_all.csv")
media   = pd.read_csv(EXPORTS / "civid_media_all.csv")
leaders = pd.read_csv(EXPORTS / "civid_leaders_all.csv")

for col in ["fatalities", "injuries", "missing"]:
    if col in events.columns:
        events[col] = pd.to_numeric(events[col], errors="coerce")
for df in (events, persons, famous):
    if "event_date" in df.columns:
        df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")

print(f"Loaded {len(events):>3} events | {len(persons):>2} persons | "
      f"{len(famous):>2} famous-victim rows | {len(media):>2} media rows")


Loaded  63 events | 14 persons |  0 famous-victim rows |  0 media rows


## 2. Dataset overview
Counts split by phase, verification, and confidence. Aggregates are labelled (`derived_is_aggregate`) so cumulative figures are not silently double-counted.

In [3]:
import pandas as pd
pd.set_option("display.max_columns", 20)

ov = pd.DataFrame({
    "Metric": ["Events (total)", "Verified", "Estimated", "Unverified", "Disputed",
               "Persons", "Famous-victim rows", "Media rows",
               "Fatalities (sum)*", "Injuries (sum)*", "Missing (sum)*",
               "Needs review", "Aggregate rows"],
    "Value": [
        len(events),
        int((events.verification_status == "verified").sum()),
        int((events.verification_status == "estimated").sum()),
        int((events.verification_status == "unverified").sum()),
        int((events.verification_status == "disputed").sum()),
        len(persons), len(famous), len(media),
        int(events.fatalities.fillna(0).sum()),
        int(events.injuries.fillna(0).sum()),
        int(events.missing.fillna(0).sum()),
        int((events.derived_needs_review == True).sum()) if "derived_needs_review" in events else 0,
        int((events.derived_is_aggregate == True).sum()) if "derived_is_aggregate" in events else 0,
    ],
})
display(ov)
print("* Sums include aggregate/cumulative rows; treat as upper-bound indicators only.")


,Metric,Value
0,Events (total),63
1,Verified,48
2,Estimated,7
3,Unverified,8
4,Disputed,0
5,Persons,14
6,Famous-victim rows,0
7,Media rows,0
8,Fatalities (sum)*,145537
9,Injuries (sum)*,183304


* Sums include aggregate/cumulative rows; treat as upper-bound indicators only.


## 3. Events by phase & year

In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if "phase" in events.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    events["phase"].value_counts().plot(kind="barh", ax=axes[0], color="#4C72B0")
    axes[0].set_title("Events by phase"); axes[0].invert_yaxis()

    yr = events["derived_year"].value_counts().sort_index()
    yr.plot(kind="bar", ax=axes[1], color="#55A868")
    axes[1].set_title("Events by year"); axes[1].set_xlabel("")
    plt.tight_layout(); plt.show()


C:\Users\Muhammad Anas\AppData\Local\Temp\ipykernel_5104\1982933411.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. Timeline (chronological, all phases)
Uses the precomputed `derived_timeline_order` for a stable ordering.

In [5]:
if {"event_date", "phase"}.issubset(events.columns):
    tl = events.dropna(subset=["event_date"]).sort_values("event_date")
    fig, ax = plt.subplots(figsize=(11, 4))
    for phase in tl["phase"].unique():
        sub = tl[tl["phase"] == phase]
        ax.scatter(sub["event_date"], [phase]*len(sub), label=phase, s=55, alpha=0.7)
    ax.set_title("Event timeline (all phases)"); ax.set_xlabel("Date")
    ax.legend(); plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()


C:\Users\Muhammad Anas\AppData\Local\Temp\ipykernel_5104\456393670.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(); plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()


## 5. Verification & confidence

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if "verification_status" in events.columns:
    vc = events["verification_status"].value_counts()
    axes[0].pie(vc.values, labels=vc.index, autopct="%1.0f%%",
               colors=["#55A868", "#DD8452", "#C44E52", "#8172B2"])
    axes[0].set_title("Verification status")
if "confidence_level" in events.columns:
    cc = events["confidence_level"].value_counts()
    axes[1].bar(cc.index, cc.values, color=["#CCB974", "#64B5CD", "#C44E52"])
    axes[1].set_title("Confidence level"); axes[1].set_ylabel("Events")
plt.tight_layout(); plt.show()


C:\Users\Muhammad Anas\AppData\Local\Temp\ipykernel_5104\1358402916.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. Victim role distribution (persons table)
Only populated where a source stated a role; `unknown` means not verifiable.

In [7]:
if "victim_role" in persons.columns:
    rc = persons["victim_role"].value_counts()
    fig, ax = plt.subplots(figsize=(9, 4))
    rc.plot(kind="barh", ax=ax, color="#8172B2")
    ax.set_title("Victim roles (person records)"); ax.set_xlabel("Persons")
    plt.tight_layout(); plt.show()
else:
    print("No victim_role column in persons export.")


C:\Users\Muhammad Anas\AppData\Local\Temp\ipykernel_5104\1752255363.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 7. Source reliability
Pulled from `exports/civid_events_all.csv` joined source fields.

In [8]:
if {"source_name", "reliability_score"}.issubset(events.columns):
    rel = (events[["source_name", "reliability_score"]]
           .dropna(subset=["reliability_score"])
           .drop_duplicates()
           .sort_values("reliability_score", ascending=False))
    display(rel)


,source_name,reliability_score
7,UN OCHA Humanitarian Situation Report | 3 July...,0.95
8,OCHA oPt Gaza Humanitarian Response SitRep #62,0.95
43,UNRWA,0.95
16,OCHA oPt,0.95
10,OCHA oPt Gaza Humanitarian Response SitRep #69,0.95
6,ICRC Sudan Operations and Humanitarian Response,0.94
49,WHO Sudan Health Cluster,0.93
45,UN OCHA Sudan Situation Report,0.91
5,Gaza Ministry of Health (MoH),0.90
0,ACLED Iran Crisis,0.89


## 8. Famous-victims & media / ethics status

In [9]:
from IPython.display import Markdown, display
print("Famous-victim rows present:", len(famous), "(empty-by-design until human-reviewed; see docs/famous_victims_policy.md)")
print("Media rows present       :", len(media))
if "contains_identifiable_person" in media.columns:
    print("Media with identifiable person:", int((media.contains_identifiable_person == True).sum()))
if len(famous) == 0:
    display(Markdown("**No famous-victim records yet.** This section is intentionally empty; "
                     "named-person data is added only through the verified, multi-source pipeline."))


Famous-victim rows present: 0 (empty-by-design until human-reviewed; see docs/famous_victims_policy.md)
Media rows present       : 0
Media with identifiable person: 0


**No famous-victim records yet.** This section is intentionally empty; named-person data is added only through the verified, multi-source pipeline.

## 8b. Verified leaders — separate table (no merges into casualty tables)
Independent leaders table for Hamas and other conflict organizations. Includes verified senior figures
reported as killed/confirmed dead/officially acknowledged. Politically sensitive records are NOT excluded.
Single-source (unverified) records are flagged `needs_review=true` and held in the review queue below.

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
if "leader_name" in leaders.columns:
    lv = leaders.copy()
    lv["death_date"] = pd.to_datetime(lv["death_date"], errors="coerce")
    print(f"Leaders total: {len(lv)} | verified: {int((lv.verification_status==\"verified\").sum())} |"
          f" unverified/review: {int((lv.needs_review==True).sum()) if \"needs_review\" in lv else 0}")
    org_counts = lv["organization"].value_counts()
    display(org_counts.rename("leaders").to_frame())
else:
    print("No leader_name column in leaders export.")


In [ ]:
from pathlib import Path as _P
from IPython.display import display, Image, Markdown

def leader_photo(r):
    lp = (r.get("image_local_path") or "").strip()
    if lp and (_P(ROOT) / lp).exists():
        return str(_P(ROOT) / lp)
    return (r.get("image_url") or "").strip() or None

if "leader_name" in leaders.columns:
    cards = leaders.sort_values("death_date", errors="coerce")
    for _, r in cards.iterrows():
        ph = leader_photo(r)
        head = f"**{r['leader_name']}** ({r['organization']} . {r['role']} . {r['leadership_level']})"
        meta = (f"Died: {r['death_date']} - {r['death_location']} ({r['death_cause']})\n"
                f"Status: {r['verification_status']} / {r['confidence_level']}"
                + (" . NEEDS REVIEW" if str(r.get('needs_review')).lower() in ('true','1') else ""))
        bio = (r.get("bio") or "")
        src = r.get("source_url") or ""
        block = head + "\n\n" + meta + "\n\n" + bio
        if ph:
            try:
                display(Image(filename=ph, width=140))
            except Exception:
                pass
        display(Markdown(block + (f"\n\n[source]({src})" if src else "")))
else:
    print("No leaders to display.")


### Review queue — unverified leader records
These were reported by a single source and not yet independently confirmed. They are kept in the
leaders table (never dropped for being politically sensitive) but must be confirmed before any publication use.

In [ ]:
if "needs_review" in leaders.columns:
    rq = leaders[leaders["needs_review"] == True]
    if len(rq):
        display(rq[["leader_id","leader_name","organization","death_status","verification_status","confidence_level","source_url","notes"]])
    else:
        display(Markdown("**No leader records currently in the review queue.**"))
else:
    print("needs_review column absent from leaders export.")


## 9. Data-quality flags (drives the review queue)

In [10]:
if "derived_ambiguity_flag" in events.columns:
    qf = pd.DataFrame({
        "Flag": ["ambiguous/unverified", "missing data", "needs review", "aggregate"],
        "Events": [
            int((events.derived_ambiguity_flag == True).sum()) if "derived_ambiguity_flag" in events else 0,
            int((events.derived_missing_data_flag == True).sum()) if "derived_missing_data_flag" in events else 0,
            int((events.derived_needs_review == True).sum()) if "derived_needs_review" in events else 0,
            int((events.derived_is_aggregate == True).sum()) if "derived_is_aggregate" in events else 0,
        ],
    })
    display(qf)
    print("Work down 'needs review' rows before any publication use.")


,Flag,Events
0,ambiguous/unverified,15
1,missing data,41
2,needs review,8
3,aggregate,14


Work down 'needs review' rows before any publication use.


---
### How to use
1. Run **Section 0** first (full-system run & check).
2. Then run Sections 1–9 top to bottom.
3. To refresh after a data update: Run All (Section 0 regenerates `exports/`).

### Notes
- Every figure reads from `exports/` regenerated by `scripts/build_exports.py`.
- `event_id` is unique per phase; the global key is `derived_global_event_key` (`phase/event_id`).
- The original `record_id` is preserved in `legacy_record_id` after renumbering.
- Licensing: code = MIT (© Muhammad Farhan); underlying data remains with each publisher — see `DATA_LICENSE.md`.
